# parameter recovery

colors trial type:


yellow = 1

orange = 2

red = 3

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["OMP_NUM_THREADS"] = "1"
import seaborn as sns
import ast
from scipy.stats import pearsonr
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import FixedLocator
from scipy.spatial.distance import euclidean
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from scipy.io import loadmat, savemat
import warnings
warnings.filterwarnings("ignore")

# read mean and std of red, orange, and yellow reward summary

In [12]:
folder = "param_recovery_2_reward_distribution"
file_path = os.path.join(folder, "reward_summary_by_color.csv")

reward_summary = pd.read_csv(file_path)

color_to_num = {
    "yellow": 1,
    "orange": 2,
    "red": 3
}

reward_stats = {
    color_to_num[row["color"]]: {
        "mean": row["mean_reward"],
        "std": row["std_reward"]
    }
    for _, row in reward_summary.iterrows()
}

# examples:
# reward_stats[3]["mean"]   # red mean
# reward_stats[3]["std"]    # red std
# reward_stats[2]["mean"]   # orange mean
# reward_stats[1]["mean"]   # yellow mean

In [13]:
outputFolderName = r"D:\Nill\data\BART\0_0_new_IED\context_modeling\param_recovery_3_simulated_fields"
inputFolderName = r"D:\Nill\data\BART\0_0_new_IED\context_modeling\param_recovery_1_modeling"


if not os.path.exists(outputFolderName):
    os.makedirs(outputFolderName)


matFiles = [f for f in os.listdir(inputFolderName) if f.endswith(".mat")]
nPatients = len(matFiles)


In [14]:

def safe_scalar(x):
    """Convert matlab-loaded scalar/0d/1-element array to python scalar."""
    arr = np.asarray(x).squeeze()
    if arr.shape == ():
        return arr.item()
    return arr

def sample_truncated_normal(mean_val, std_val, low=None, high=None, max_tries=10000):
    """
    Sample from a normal distribution with optional lower/upper bounds.
    Falls back safely if repeated draws fail.
    """
    if (not np.isfinite(std_val)) or std_val <= 0:
        val = mean_val
        if low is not None:
            val = max(val, low)
        if high is not None:
            val = min(val, high)
        return float(max(0, val))

    for _ in range(max_tries):
        val = np.random.normal(mean_val, std_val)
        if low is not None and val < low:
            continue
        if high is not None and val > high:
            continue
        return float(max(0, val))

    # fallback
    val = mean_val
    if low is not None:
        val = max(val, low)
    if high is not None:
        val = min(val, high)
    return float(max(0, val))

In [15]:

for pt in range(nPatients):
# for pt in range(3):

    fileName = matFiles[pt]

    ptID = os.path.splitext(fileName)[0]
    ptID = ptID.replace("_TDdataParamRecovery", "")

    print(f"processing pt {pt+1}/{nPatients}: {ptID}")

    matFile = os.path.join(inputFolderName, fileName)
    mat = loadmat(matFile, struct_as_record=False, squeeze_me=True)

    TDdataParamRecovery = mat["TDdataParamRecovery"]

    alphas = np.asarray(TDdataParamRecovery.a, dtype=float).squeeze()
    bestAlphaPos = float(safe_scalar(TDdataParamRecovery.bestAlphaPos))
    bestAlphaNeg = float(safe_scalar(TDdataParamRecovery.bestAlphaNeg))

    bestAlphaPosIdx = int(np.where(np.isclose(alphas, bestAlphaPos))[0][0])
    bestAlphaNegIdx = int(np.where(np.isclose(alphas, bestAlphaNeg))[0][0])

    result = np.asarray(TDdataParamRecovery.result).astype(str).squeeze()
    reward = np.asarray(TDdataParamRecovery.Reward, dtype=float).squeeze()
    isControl = np.asarray(TDdataParamRecovery.is_control).astype(int).squeeze()
    trial_type = np.asarray(TDdataParamRecovery.trial_type).astype(int).squeeze()



    RPE_all = np.asarray(TDdataParamRecovery.RPE, dtype=float)
    expectedReward_all = np.asarray(TDdataParamRecovery.expectedReward, dtype=float)

    bestRPE = np.asarray(RPE_all[bestAlphaPosIdx, bestAlphaNegIdx, :], dtype=float).squeeze()
    best_expected_reward = np.asarray(expectedReward_all[bestAlphaPosIdx, bestAlphaNegIdx, :], dtype=float).squeeze()


    nTrials = len(TDdataParamRecovery.scoreVec)-1

    result = result[:nTrials]
    reward = reward[:nTrials]
    isControl = isControl[:nTrials]
    trial_type = trial_type[:nTrials]
    bestRPE = bestRPE[:nTrials]
    best_expected_reward = best_expected_reward[:nTrials]

    resultSimulated = np.empty(nTrials, dtype=object)
    rewardSimulated = np.full(nTrials, np.nan)

    neg_rpe_vals = np.abs(bestRPE[bestRPE < 0])
    neg_rpe_scale = np.nanmedian(neg_rpe_vals) if len(neg_rpe_vals) > 0 else 1.0
    if not np.isfinite(neg_rpe_scale) or neg_rpe_scale == 0:
        neg_rpe_scale = 1.0

    for trial in range(nTrials):

        # control trials: keep identical
        if isControl[trial] == 1:
            resultSimulated[trial] = result[trial]
            rewardSimulated[trial] = reward[trial]
            continue

        rpe = bestRPE[trial]
        this_result = result[trial]
        this_reward = reward[trial]
        this_trial_type = trial_type[trial]

        if this_trial_type not in reward_stats:
            resultSimulated[trial] = this_result
            rewardSimulated[trial] = this_reward
            continue

        mean_reward = reward_stats[this_trial_type]["mean"]
        std_reward = reward_stats[this_trial_type]["std"]

        # RPE ~ 0 : keep same
        if np.isclose(rpe, 0):
            resultSimulated[trial] = this_result
            rewardSimulated[trial] = this_reward

        # positive RPE:
        # actual reward > expected reward
        # model would tend to stop earlier / more conservatively
        elif rpe > 0:

            resultSimulated[trial] = "banked"

            if this_result == "banked":
                simulated_reward = sample_truncated_normal(
                    mean_val=min(this_reward, mean_reward),
                    std_val=std_reward,
                    low=0,
                    high=this_reward
                )
            else:
                upper_bound = max(0, best_expected_reward[trial])
                simulated_reward = sample_truncated_normal(
                    mean_val=min(mean_reward, upper_bound),
                    std_val=std_reward,
                    low=0,
                    high=upper_bound
                )

            rewardSimulated[trial] = simulated_reward

        # negative RPE:
        # actual reward < expected reward
        # model would tend to keep pumping more
        else:

            if this_result == "popped":
                resultSimulated[trial] = "popped"
                rewardSimulated[trial] = 0.0

            else:
                pop_prob = 1.0 - np.exp(-abs(rpe) / neg_rpe_scale)
                pop_prob = np.clip(pop_prob, 0.05, 0.95)

                if np.random.rand() < pop_prob:
                    resultSimulated[trial] = "popped"
                    rewardSimulated[trial] = 0.0
                else:
                    simulated_reward = sample_truncated_normal(
                        mean_val=max(this_reward, mean_reward),
                        std_val=std_reward,
                        low=this_reward,
                        high=None
                    )
                    resultSimulated[trial] = "banked"
                    rewardSimulated[trial] = simulated_reward

    TDdataParamRecovery.resultSimulated = np.array(resultSimulated, dtype=object)
    TDdataParamRecovery.rewardSimulated = np.array(rewardSimulated, dtype=float)
    TDdataParamRecovery.scoreVec = np.array(TDdataParamRecovery.scoreVec, dtype=float)

    # save to output folder
    outFile = os.path.join(outputFolderName, f"{ptID}_TDdataParamRecovery.mat")
    savemat(outFile, {"TDdataParamRecovery": TDdataParamRecovery}, do_compression=True)

processing pt 1/71: 201810
processing pt 2/71: 201811
processing pt 3/71: 201901
processing pt 4/71: 201902r
processing pt 5/71: 201902
processing pt 6/71: 201903
processing pt 7/71: 201905
processing pt 8/71: 201909
processing pt 9/71: 201910
processing pt 10/71: 201911
processing pt 11/71: 201914
processing pt 12/71: 201915
processing pt 13/71: 202001
processing pt 14/71: 202002
processing pt 15/71: 202003
processing pt 16/71: 202004
processing pt 17/71: 202005
processing pt 18/71: 202006u
processing pt 19/71: 202007
processing pt 20/71: 202008
processing pt 21/71: 202011
processing pt 22/71: 202014
processing pt 23/71: 202015
processing pt 24/71: 202016
processing pt 25/71: 202105
processing pt 26/71: 202107
processing pt 27/71: 202110
processing pt 28/71: 202114
processing pt 29/71: 202117
processing pt 30/71: 202118
processing pt 31/71: 202201
processing pt 32/71: 202202
processing pt 33/71: 202205
processing pt 34/71: 202207
processing pt 35/71: 202208
processing pt 36/71: 202209

# debug

In [16]:
fields = [f for f in dir(TDdataParamRecovery) if not f.startswith('_')]
print(fields)

['RPE', 'Reward', 'a', 'bestAlphaNeg', 'bestAlphaPos', 'bestAnIdx', 'bestApIdx', 'bestFitScore', 'bestInverseTemperature', 'bestLogLik', 'expectedReward', 'fitScoreRSTD', 'inflate_time', 'inverseTemperatureRSTD', 'is_control', 'logLikRSTD', 'logPriorAlphaNeg', 'logPriorAlphaPos', 'points', 'pointsMinusReward', 'result', 'resultSimulated', 'rewardSimulated', 'scoreVec', 'trialTypes', 'trial_type']


In [17]:
np.shape(isControl)

(236,)